In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
units         = np.load('../Data/units.npy',allow_pickle=True)
info          = np.load('../Data/trial_corrInfo.npy',allow_pickle=True)
reaction      = np.load('../Data/reaction.npy',allow_pickle=True)
letters       = np.load('../Data/letters_sep.npy',allow_pickle=True)
probes        = np.load('../Data/probes_sep.npy',allow_pickle=True)
uniqueLetters = np.array([p for p in np.unique(np.concatenate(letters)) if p not in ['Q', 'W', 'X']])
subj          = np.load('../Data/subjs_sep.npy',allow_pickle=True)
subj          = np.concatenate(subj,axis=0)
uniqueSub     = np.unique(subj)
len(subj)

2613

In [3]:
allL = np.concatenate(letters,axis=0)
allL = [[si for si in s if si not in ['X','Q','W']] for s in allL ]

allP = np.concatenate(probes,axis=0)
allP = [[si for si in s if si not in ['X','Q','W']] for s in allP ]

infoALL = np.concatenate(info)
infoALL = infoALL[infoALL[:,1]==1]

respALL = infoALL[:,-1]

In [4]:
global_letter_acc = []
L  = 15
T1 = 2
T2 = 16
T  = T2-T1
for k in range(50):
    al = []
    all_L = np.load(f'../single_letter_50_{k}.npy')
    for letter_index in range(L):
        al.append(np.diag(all_L[letter_index]))
    al = np.array(al)
    global_letter_acc.append(al)

global_letter_acc = np.array(global_letter_acc)  # shape: (50, L, n_times)

# Compute global last-step accuracy like your code
global_last = np.array([
    np.cumsum(global_letter_acc[:, i, T1:T2], axis=-1)[:, -1] / T
    for i in range(L)
])  # shape: (L, 50)

# Mean global decoding accuracy per letter                     ### NEW
global_mean_per_letter = np.median(global_last, axis=1)          # shape: (L,)  ### NEW

In [5]:
dictDEC = {}
for i in range(len(uniqueLetters)):
    dictDEC[uniqueLetters[i]]= global_mean_per_letter[i]
dictDEC

{'B': 0.5725,
 'C': 0.5549999999999999,
 'D': 0.5460714285714285,
 'F': 0.5964285714285714,
 'G': 0.5585714285714286,
 'H': 0.5,
 'K': 0.5214285714285716,
 'L': 0.48964285714285716,
 'N': 0.5449999999999999,
 'P': 0.4782142857142857,
 'R': 0.5399999999999999,
 'S': 0.49535714285714283,
 'T': 0.5082142857142857,
 'V': 0.5467857142857143,
 'Z': 0.5053571428571428}

In [8]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rows = []

uniqueSub = np.array(uniqueSub)  # as before

for i in range(len(infoALL)):
    letter_set   = allL[i]
    probe_single = allP[i]

    if len(probe_single):
        probe_single = probe_single[0]

        info_single  = infoALL[i]
        inout        = info_single[3]      # 51 or 52
        rt           = info_single[4]
        set_size     = info_single[0]      # 4, 6 or 8

        # predictors
        predicted  = np.mean([dictDEC[l] for l in letter_set])
        #pred_probe = np.mean([dictDEC[l] for l in probe_single])  # mean of 1 item is itself

        # original subject ID
        subject_id = subj[i]

        rows.append({
            "rt": rt,
            "predicted": predicted,
            "inout": int(inout),          # 51/52
            "size": int(set_size),        # 6/8
            "subject": subject_id,
            "probe": probe_single
        })

# Build DataFrame
df = pd.DataFrame(rows)

# Make sure predictors that are categorical are treated as such
df["inout"] = df["inout"].astype("category")
df["size"]  = df["size"].astype("category")
df["probe"] = df["probe"].astype("category")

# ---------------------------------------------------------------------
# NEW: make 'G' the reference probe (baseline) for C(probe)
# ---------------------------------------------------------------------
df["probe"] = df["probe"].cat.remove_unused_categories()

if "G" in df["probe"].cat.categories:
    new_order = ["G"] + [c for c in df["probe"].cat.categories if c != "G"]
    df["probe"] = df["probe"].cat.reorder_categories(new_order, ordered=True)
else:
    print("Warning: 'G' is not present in probe categories; default reference will be used.")

# Mixed model: RT ~ predicted + inout (51 vs 52) + size (6 vs 8) + probe (with 'G' as reference)
# Random intercepts for subject
formula = (
    "rt ~ predicted "
    "+ C(inout) "
    "+ C(size) "
    "+ C(probe)"
)

model = smf.mixedlm(
    formula,
    df,
    groups=df["subject"],
    re_formula="1"        # random intercept only
)

res = model.fit(reml=False)  # ML; use reml=True if you prefer REML

# ---------------------------------------------------------------------
# Build clean fixed-effects table: Coef, CI [low, high], z, p
# ---------------------------------------------------------------------
fe_params = res.fe_params
fe_se     = res.bse_fe
conf_int  = res.conf_int().loc[fe_params.index]
fe_pvals  = res.pvalues.loc[fe_params.index]

# z-value (coefficient / standard error)
z_values = fe_params / fe_se

# CI formatted in square brackets
ci_strings = conf_int.apply(
    lambda r: f"[{r[0]:.3f}, {r[1]:.3f}]",
    axis=1
)

results_table = pd.DataFrame({
    "Coefficient": fe_params.round(3),
    "CI": ci_strings,
    "z": z_values.round(3),
    "p-value": fe_pvals.apply(lambda p: f"{p:.2e}")
})

from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------------
# Identify probe effects and size effects
# ---------------------------------------------------------------------
probe_mask = fe_pvals.index.str.startswith("C(probe)")
size_mask  = fe_pvals.index.str.startswith("C(size)")

# Extract subsets
probe_pvals = fe_pvals[probe_mask]
size_pvals  = fe_pvals[size_mask]

# ---------------------------------------------------------------------
# Apply FDR separately for probes and size
# ---------------------------------------------------------------------
_, probe_pvals_fdr, _, _ = multipletests(probe_pvals, method="fdr_bh")
_, size_pvals_fdr,  _, _ = multipletests(size_pvals,  method="fdr_bh")

# Put corrected p-values into table
results_table["p-FDR"] = ""

results_table.loc[probe_mask, "p-FDR"] = [
    f"{p:.2e}" for p in probe_pvals_fdr
]

results_table.loc[size_mask, "p-FDR"] = [
    f"{p:.2e}" for p in size_pvals_fdr
]

In [ ]:
#import re
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# ── Extract fixed effects ─────────────────────────────────────────────────
fe_params = res.fe_params
fe_se     = res.bse_fe
conf_int  = res.conf_int().loc[fe_params.index]
fe_pvals  = res.pvalues.loc[fe_params.index]
z_values  = fe_params / fe_se

ci_strings = conf_int.apply(
    lambda r: f'[{r[0]:.3f}, {r[1]:.3f}]', axis=1
)

# ── FDR correction within probe and size families ─────────────────────────
probe_mask = fe_pvals.index.str.startswith('C(probe)')
size_mask  = fe_pvals.index.str.startswith('C(size)')

_, probe_fdr, _, _ = multipletests(fe_pvals[probe_mask], method='fdr_bh')
_, size_fdr,  _, _ = multipletests(fe_pvals[size_mask],  method='fdr_bh')

# Start with raw p-values, overwrite corrected families in place
pval_display = fe_pvals.copy()
pval_display[probe_mask] = probe_fdr
pval_display[size_mask]  = size_fdr

def fmt_p(p):
    return '< .001' if p < .001 else f'{p:.3f}'

# ── Build results table ───────────────────────────────────────────────────
results_table = pd.DataFrame({
    'Coefficient': fe_params.round(3),
    'CI':          ci_strings,
    'z-value':     z_values.round(3),
    'p-value':     pval_display.apply(fmt_p),
}, index=fe_params.index)

# ── Rename terms to human-readable labels ─────────────────────────────────
RENAME = {
    'Intercept':        'Intercept',
    'predicted_z':      'Decoding accuracy',
    'phon_sim_z':       'Phonological similarity (string)',
    'probe_phon_sim_z': 'Phonological similarity (probe–string)',
    'serial_pos_z':     'Serial position (IN trials)',
}

def clean_term(term):
    if term in RENAME:
        return RENAME[term]
    m = re.match(r'C\((?P<var>[^)]+)\)\[T\.(?P<lvl>.+)\]$', term)
    if m:
        var, lvl = m.group('var'), m.group('lvl')
        ref = df[var].cat.categories[0]
        return f'{lvl} vs. {ref}'
    return term

results_table.index = [clean_term(t) for t in results_table.index]

# ── Reorder: key predictors first, covariates after ───────────────────────
PRIORITY = [
    'Intercept',
    'Decoding accuracy',
    'Phonological similarity (string)',
    'Phonological similarity (probe–string)',
    'Serial position (IN trials)',
]
idx_order     = PRIORITY + [x for x in results_table.index if x not in PRIORITY]
results_table = results_table.reindex(
    [x for x in idx_order if x in results_table.index]
)

# ── Export ────────────────────────────────────────────────────────────────
results_table.to_excel('Mixed-effect_RT_revised.xlsx')
results_table

In [9]:
df.to_csv('table_df.csv')